In [ ]:
!pip install -q -U transformers datasets scikit-learn

In [ ]:
import glob, json, threading
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import Counter

DEVICE0   = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEVICE1   = torch.device("cuda:1" if torch.cuda.device_count() > 1 else DEVICE0)
MAX_LEN   = 128
BATCH     = 512   # larger batch — tokenization is no longer the bottleneck
OUT_DIR   = Path("/kaggle/working")
print(f"Devices: {DEVICE0}, {DEVICE1}")
print(f"GPU count: {torch.cuda.device_count()}")

In [ ]:
# ── Find input files ──────────────────────────────────────────────────────
def find_input_file(*names):
    for name in names:
        for pattern in [f"/kaggle/input/**/{name}", f"/kaggle/input/datasets/kevinnchan/**/{name}"]:
            matches = glob.glob(pattern, recursive=True)
            if matches:
                return Path(matches[0])
        local = Path(name)
        if local.exists(): return local
    raise FileNotFoundError(f"Cannot find any of {names}")

In [ ]:
# ── Load all 4 models (find by axis name in id2label.json, slug-agnostic) ─
def load_model(axis_name, device):
    candidates = glob.glob("/kaggle/input/**/id2label.json", recursive=True)
    for c in candidates:
        try:
            with open(c) as f: meta = json.load(f)
            if meta.get("axis") != axis_name: continue
            model_dir = Path(c).parent
            id2label  = {int(k): v for k, v in meta["id2label"].items()}
            label2id  = {v: k for k, v in id2label.items()}
            tok   = AutoTokenizer.from_pretrained(str(model_dir))
            model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
            model.eval().to(device)
            print(f"Loaded axis={axis_name} from {model_dir}  labels={id2label}  device={device}")
            return tok, model, id2label, label2id
        except Exception:
            continue
    raise FileNotFoundError(f"No model found for axis: {axis_name}")

# Load Stage 1 models first to get tokenizer — all 4 share the same base tokenizer
tok1a, model1a, id2label1a, label2id1a = load_model("buyin_relevance",  DEVICE0)
tok1b, model1b, id2label1b, label2id1b = load_model("stance_relevance", DEVICE1)
tok2a, model2a, id2label2a, label2id2a = load_model("buyin_direction",  DEVICE0)
tok2b, model2b, id2label2b, label2id2b = load_model("stance_direction", DEVICE1)

In [ ]:
# ── Load chunk text ───────────────────────────────────────────────────────
def load_chunks():
    cc_path = find_input_file("comments_chunks.parquet")
    sc_path = find_input_file("submissions_chunks.parquet")
    cc = pd.read_parquet(cc_path, columns=["chunk_id", "doc_id", "text"])
    sc = pd.read_parquet(sc_path, columns=["chunk_id", "doc_id", "text"])
    df = pd.concat([cc, sc], ignore_index=True).drop_duplicates("chunk_id")
    df["text"] = df["text"].fillna("").astype(str).str.strip()
    df = df[df["text"].str.len() > 5].reset_index(drop=True)
    print(f"Chunks loaded: {len(df):,}")
    return df

chunks    = load_chunks()
texts     = chunks["text"].tolist()
chunk_ids = chunks["chunk_id"].tolist()
N         = len(texts)

In [ ]:
# ── Pre-tokenize ONCE (all models share same SingBERT tokenizer) ──────────
# Tokenize in batches to avoid OOM on 727k strings
import time
print(f"Pre-tokenizing {N:,} chunks (batch 10k)...")
t0 = time.time()

TOK_BATCH = 10_000
all_input_ids      = []
all_attention_mask = []
all_token_type_ids = []

for start in range(0, N, TOK_BATCH):
    batch_texts = texts[start:start + TOK_BATCH]
    enc = tok1a(batch_texts, truncation=True, max_length=MAX_LEN,
                padding="max_length", return_tensors="pt")
    all_input_ids.append(enc["input_ids"])
    all_attention_mask.append(enc["attention_mask"])
    if "token_type_ids" in enc:
        all_token_type_ids.append(enc["token_type_ids"])
    if (start // TOK_BATCH) % 5 == 0:
        print(f"  tokenized {start+len(batch_texts):,}/{N:,}", end="\r")

input_ids      = torch.cat(all_input_ids,      dim=0)
attention_mask = torch.cat(all_attention_mask, dim=0)
has_tti        = len(all_token_type_ids) > 0
if has_tti:
    token_type_ids = torch.cat(all_token_type_ids, dim=0)

print(f"\nTokenization done in {time.time()-t0:.0f}s  shape={input_ids.shape}")
# Shared read-only tensors — no copy needed per thread

In [ ]:
# ── Inference helper (TensorDataset — no per-item tokenization) ───────────
@torch.no_grad()
def run_inference(indices, model, device, batch_size=BATCH, desc=""):
    """Run model on a subset of the pre-tokenized corpus."""
    idx_t = torch.tensor(indices, dtype=torch.long)
    iids  = input_ids[idx_t]
    amask = attention_mask[idx_t]
    if has_tti:
        tti   = token_type_ids[idx_t]
        ds = TensorDataset(iids, amask, tti)
    else:
        ds = TensorDataset(iids, amask)

    loader = DataLoader(ds, batch_size=batch_size, num_workers=0, pin_memory=True)
    all_probs = []
    n = len(indices)
    for i, batch in enumerate(loader):
        if has_tti:
            iid, am, tti_b = [t.to(device) for t in batch]
            out = model(input_ids=iid, attention_mask=am, token_type_ids=tti_b)
        else:
            iid, am = [t.to(device) for t in batch]
            out = model(input_ids=iid, attention_mask=am)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        if i % 100 == 0:
            print(f"  {desc} {min((i+1)*batch_size, n):,}/{n:,}", end="\r")
    print(f"  {desc} done — {n:,} chunks                 ")
    return np.concatenate(all_probs, axis=0)

In [ ]:
# ── Parallel execution: buyin branch (GPU0) and stance branch (GPU1) ──────
all_indices = list(range(N))
results     = {}   # shared dict — each thread writes its own keys
errors      = []

def buyin_branch():
    try:
        import time
        t = time.time()
        print("[BUYIN]  Stage 1a: buyin relevance ...")
        p1a = run_inference(all_indices, model1a, DEVICE0, desc="1a")
        print(f"[BUYIN]  Stage 1a done in {time.time()-t:.0f}s")

        rel_col     = label2id1a.get("relevant",     1)
        not_col     = label2id1a.get("not_relevant", 0)
        is_relevant = p1a[:, rel_col] >= 0.5
        rel_indices = [i for i in all_indices if is_relevant[i]]
        print(f"[BUYIN]  Relevant: {len(rel_indices):,}/{N:,} ({len(rel_indices)/N:.1%})")

        print("[BUYIN]  Stage 2a: committed vs uncommitted ...")
        t = time.time()
        p2a = run_inference(rel_indices, model2a, DEVICE0, desc="2a")
        print(f"[BUYIN]  Stage 2a done in {time.time()-t:.0f}s")

        com_col = label2id2a.get("committed",   0)
        unc_col = label2id2a.get("uncommitted", 1)

        prob_committed   = np.zeros(N)
        prob_uncommitted = np.zeros(N)
        prob_neutral     = np.ones(N)
        buyin_label      = ["neutral"] * N

        for arr_i, corp_i in enumerate(rel_indices):
            prob_committed[corp_i]   = p2a[arr_i, com_col]
            prob_uncommitted[corp_i] = p2a[arr_i, unc_col]
            prob_neutral[corp_i]     = 0.0
            buyin_label[corp_i] = "committed" if p2a[arr_i, com_col] >= p2a[arr_i, unc_col] else "uncommitted"

        # Non-relevant: spread stage-1a rel prob equally as soft signal
        for i in range(N):
            if not is_relevant[i]:
                prob_neutral[i]     = p1a[i, not_col]
                prob_committed[i]   = p1a[i, rel_col] * 0.5
                prob_uncommitted[i] = p1a[i, rel_col] * 0.5

        results["buyin"] = (buyin_label, prob_committed, prob_uncommitted, prob_neutral)
        print(f"[BUYIN]  Label dist: {Counter(buyin_label)}")

        # Save intermediate in case stance branch or assembly fails
        pd.DataFrame({
            "chunk_id": chunk_ids,
            "buyin_label": buyin_label,
            "prob_buyin_committed":   prob_committed.round(5),
            "prob_buyin_uncommitted": prob_uncommitted.round(5),
            "prob_buyin_neutral":     prob_neutral.round(5),
        }).to_parquet(OUT_DIR / "_buyin_intermediate.parquet", index=False)
        print("[BUYIN]  Intermediate saved.")
    except Exception as e:
        errors.append(("buyin", e))
        raise


def stance_branch():
    try:
        import time
        t = time.time()
        print("[STANCE] Stage 1b: stance relevance ...")
        p1b = run_inference(all_indices, model1b, DEVICE1, desc="1b")
        print(f"[STANCE] Stage 1b done in {time.time()-t:.0f}s")

        has_col    = label2id1b.get("has_stance", 1)
        no_col     = label2id1b.get("no_stance",  0)
        has_stance = p1b[:, has_col] >= 0.5
        st_indices = [i for i in all_indices if has_stance[i]]
        print(f"[STANCE] Has stance: {len(st_indices):,}/{N:,} ({len(st_indices)/N:.1%})")

        print("[STANCE] Stage 2b: supportive vs critical ...")
        t = time.time()
        p2b = run_inference(st_indices, model2b, DEVICE1, desc="2b")
        print(f"[STANCE] Stage 2b done in {time.time()-t:.0f}s")

        sup_col  = label2id2b.get("supportive", 0)
        crit_col = label2id2b.get("critical",   1)

        prob_supportive = np.zeros(N)
        prob_critical   = np.zeros(N)
        prob_neutral    = np.ones(N)
        stance_label    = ["neutral"] * N

        for arr_i, corp_i in enumerate(st_indices):
            prob_supportive[corp_i] = p2b[arr_i, sup_col]
            prob_critical[corp_i]   = p2b[arr_i, crit_col]
            prob_neutral[corp_i]    = 0.0
            stance_label[corp_i] = "supportive" if p2b[arr_i, sup_col] >= p2b[arr_i, crit_col] else "critical"

        for i in range(N):
            if not has_stance[i]:
                prob_neutral[i]     = p1b[i, no_col]
                prob_supportive[i]  = p1b[i, has_col] * 0.5
                prob_critical[i]    = p1b[i, has_col] * 0.5

        results["stance"] = (stance_label, prob_supportive, prob_critical, prob_neutral)
        print(f"[STANCE] Label dist: {Counter(stance_label)}")

        pd.DataFrame({
            "chunk_id": chunk_ids,
            "stance_label": stance_label,
            "prob_stance_supportive": prob_supportive.round(5),
            "prob_stance_critical":   prob_critical.round(5),
            "prob_stance_neutral":    prob_neutral.round(5),
        }).to_parquet(OUT_DIR / "_stance_intermediate.parquet", index=False)
        print("[STANCE] Intermediate saved.")
    except Exception as e:
        errors.append(("stance", e))
        raise


import time
t_total = time.time()
print("=" * 60)
print("Launching buyin and stance branches in parallel...")
print("=" * 60)

ta = threading.Thread(target=buyin_branch,  name="buyin",  daemon=False)
tb = threading.Thread(target=stance_branch, name="stance", daemon=False)
ta.start()
tb.start()
ta.join()
tb.join()

if errors:
    for name, e in errors:
        print(f"ERROR in {name} branch: {e}")
    raise RuntimeError("One or more branches failed — check above.")

print(f"\nBoth branches complete. Total elapsed: {time.time()-t_total:.0f}s")

In [ ]:
# ── Assemble output parquet ───────────────────────────────────────────────
buyin_label,  prob_b_com, prob_b_unc, prob_b_neu = results["buyin"]
stance_label, prob_s_sup, prob_s_cri, prob_s_neu = results["stance"]

result = pd.DataFrame({
    "chunk_id":               chunk_ids,
    "buyin_label":            buyin_label,
    "stance_label":           stance_label,
    "prob_buyin_committed":   prob_b_com.round(5),
    "prob_buyin_uncommitted": prob_b_unc.round(5),
    "prob_buyin_neutral":     prob_b_neu.round(5),
    "prob_stance_supportive": prob_s_sup.round(5),
    "prob_stance_critical":   prob_s_cri.round(5),
    "prob_stance_neutral":    prob_s_neu.round(5),
})

out_path = OUT_DIR / "chunk_commitment_cascade.parquet"
result.to_parquet(out_path, index=False)
print(f"Saved → {out_path}  ({result.shape})")
print(f"Buyin:  {result['buyin_label'].value_counts().to_dict()}")
print(f"Stance: {result['stance_label'].value_counts().to_dict()}")

In [ ]:
# ── Quick eval against testset ────────────────────────────────────────────
ts_path = find_input_file("commitment_testset.parquet")
ts      = pd.read_parquet(ts_path)
merged  = ts.merge(result, on="chunk_id", how="inner")
print(f"Testset rows with predictions: {len(merged)}")

from sklearn.metrics import classification_report
for gold_col, pred_col, name in [
    ("human_label",  "buyin_label",  "BUYIN"),
    ("human_stance", "stance_label", "STANCE"),
]:
    sub = merged[merged[gold_col].notna()]
    print(f"\n{'='*60}")
    print(f"  CASCADE {name} — n={len(sub)}")
    print(classification_report(sub[gold_col], sub[pred_col], digits=3))